In [1]:
!pip install -q datasets

from datasets import load_dataset
import pandas as pd

ds = load_dataset("meal-bbang/Korean_message")
df = ds["train"].to_pandas()

print(df.shape)
print(df["class"].value_counts())

print("\n=== class 3 예시 ===")
for t in df[df["class"] == 3]["content"].sample(10, random_state=0):
    print("-", t[:150].replace("\n", " "))
    print()

README.md:   0%|          | 0.00/869 [00:00<?, ?B/s]

lgaidataset.csv:   0%|          | 0.00/2.84M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19009 [00:00<?, ? examples/s]

(19009, 3)
class
2    10531
1     5778
3     2700
Name: count, dtype: int64

=== class 3 예시 ===
- [web발신] [쿠팡] 로캣배송 의류 문 앞(으)로 배송 완료했습니다. 사진: url: https://coupang.com/photo212

- [한진택배] 상품 집하 안내 안녕하세요 고객님. 한진택배입니다. 오늘 20:00~22:00 사이에 상품을 집하 할 예정입니다.  포장불량, 박스당 200만원 초과 상품, 현금성 상품 및 대체불가능한 상품(중요서류 원본, 골동품)등은 접수제한되오니 양해 부탁드립니다.  

- 롯데택배 집배송 안내 딩동 이마트 고객님! 진심을 다하는 롯데택배입니다.  고객님의 소중한 상품이 도착되었다는 소식을 알려드립니다.  보내는 분(곳): 이마트 상품명: 뷰티사 뷰티제품 운송장번호: LT00000281 배송지: 대구광역시 수성구 범어동 77 배송예정시간:

- 최강민 고객님께서 주문하신 상품을 배송완료 하였습니다. CJ대한통운 택배를 이용해주셔서 감사합니다.  보내는곳: (주)런이십일 인수자(장소): 관리실 운송장번호: CJ00000251 배송사원: 010-3339-7621

- [SLX택배] 박민수 고객님, 안녕하세요. SLX택배 입니다. 주문하신 알라딘 상품을 안전하게 배송해드렸습니다. 감사합니다.  ▶ 받는분: 박민수 ▶ 인수자(위탁장소): 문 앞 ▶ 보내는분: 알라딘 ▶ 운송장번호: SLX00000215 ▶ 배송원연락처: 010-2925

- [web발신] [쿠팡] 로캣배송 헤드폰 문 앞(으)로 배송 완료했습니다. 사진: url: https://coupang.com/photo109

- 이영희 고객님께서 주문하신 상품을 배송완료 하였습니다. CJ대한통운 택배를 이용해주셔서 감사합니다.  보내는곳: (주)해피스토어 인수자(장소): 문 앞 운송장번호: CJ00000087 배송사원: 010-2822-8826

- [SLX택배] 송하나 고객님, 안녕하세요. SL

라벨 정리 + 학습/테스트 나누기

In [2]:
from sklearn.model_selection import train_test_split

df["label"] = (df["class"] == 2).astype(int)
print(df["label"].value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    df["content"], df["label"],
    test_size=0.2, random_state=42, stratify=df["label"]
)
print("학습:", len(X_train), "/ 테스트:", len(X_test))

label
1    10531
0     8478
Name: count, dtype: int64
학습: 15207 / 테스트: 3802


Baseline (TF-IDF + 로지스틱 회귀)

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=50000)
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred, target_names=["정상", "사기"], digits=3))

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print("정상 오탐률(FPR):", round(fp / (fp + tn), 3))

              precision    recall  f1-score   support

          정상      0.995     1.000     0.998      1696
          사기      1.000     0.996     0.998      2106

    accuracy                          0.998      3802
   macro avg      0.998     0.998     0.998      3802
weighted avg      0.998     0.998     0.998      3802

정상 오탐률(FPR): 0.0


중복 제거 후 재실험

In [4]:
import re

def norm(t):
    t = re.sub(r"http\S+|www\.\S+", "URL", t)
    t = re.sub(r"\d", "0", t)
    t = re.sub(r"\s+", "", t)
    return t

df["norm"] = df["content"].apply(norm)
dd = df.drop_duplicates("norm")
print("전체:", len(df), "→ 중복 제거 후:", len(dd))
print(dd["label"].value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    dd["content"], dd["label"], test_size=0.2, random_state=42, stratify=dd["label"]
)
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=50000)
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)
model = LogisticRegression(max_iter=1000).fit(X_train_vec, y_train)
y_pred = model.predict(X_test_vec)

print(classification_report(y_test, y_pred, target_names=["정상", "사기"], digits=3))
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print("정상 오탐률(FPR):", round(fp / (fp + tn), 3))

print("\n=== 틀린 문자 ===")
for t, y, p in zip(X_test, y_test, y_pred):
    if y != p:
        print(f"[정답 {'사기' if y else '정상'} / 예측 {'사기' if p else '정상'}]", t[:100].replace("\n", " "))

전체: 19009 → 중복 제거 후: 14011
label
1    7394
0    6617
Name: count, dtype: int64
              precision    recall  f1-score   support

          정상      0.992     0.999     0.995      1324
          사기      0.999     0.993     0.996      1479

    accuracy                          0.996      2803
   macro avg      0.996     0.996     0.996      2803
weighted avg      0.996     0.996     0.996      2803

정상 오탐률(FPR): 0.001

=== 틀린 문자 ===
[정답 사기 / 예측 정상] [C J대한통운]택 배배송 불 가"주"소불완 전함,즉시변경 부탁드 립 니 다.{ c11.kr/I50i }
[정답 사기 / 예측 정상] 모두가 할 수 있지만 누구나 할 수 없는 필승전략 분석 픽  최반장.net
[정답 사기 / 예측 정상] 편의점에서 영수증으로 출력되는 온라인 문화상품권 사줄수 있어?  나 지금 폰 매장에 맡겨둬서 이체가 안돼서 그래.. 저녁에 폰 찾는데로 바로 돈 보낼꼐  편의점에 가서 오만원권으로
[정답 사기 / 예측 정상] [빙그레]빙 그 레 가 돌 아 왔 습 니 다다 시 모 시 러 왔 습 니 다빙그레.ⓒoⓜ
[정답 사기 / 예측 정상] +:☆:+릴☆짱+:☆:+오늘 하루도 열씨미좋은 일만가득하길멋진 하루 시작하세요~!zag➄➄➆.ℂòℳ
[정답 사기 / 예측 정상] [한진택배]5/22 고객택배는 현관 에보관합니다.사진 보기 dll.kr.Bs
[정답 사기 / 예측 정상] 오늘 저녁에 같이 치킨먹으면서 대한민국 vs 이란 응원해요 '*만' 포인트 지원할게요 주소창에 성대리 쩜 컴
[정답 사기 / 예측 정상] 쓰나미를 예측하는 고수들의 방어적 조합

글자 깨뜨리기 테스트

In [5]:
import random
random.seed(0)

def noisy(t, p=0.3):
    out = []
    for ch in t:
        out.append(ch)
        if ch != " " and random.random() < p:
            out.append(random.choice([" ", ".", "*", "_"]))
    return "".join(out)

X_test_noisy = X_test.apply(noisy)
print("예시:", X_test_noisy.iloc[0][:80])

y_pred_noisy = model.predict(vec.transform(X_test_noisy))
print(classification_report(y_test, y_pred_noisy, target_names=["정상", "사기"], digits=3))

예시: 아듀 쪽바_리 역사속으로 수장 되길. ㅎㅎㅎ
              precision    recall  f1-score   support

          정상      0.983     0.995     0.989      1324
          사기      0.996     0.984     0.990      1479

    accuracy                          0.990      2803
   macro avg      0.989     0.990     0.990      2803
weighted avg      0.990     0.990     0.990      2803



In [6]:
hard = pd.read_csv("hard_test.csv")
hard_pred = model.predict(vec.transform(hard["content"]))

print(classification_report(hard["label"], hard_pred, target_names=["정상", "사기"], digits=3))

hard["pred"] = hard_pred
print("\n=== 유형별 정답률 ===")
print((hard["pred"] == hard["label"]).groupby(hard["type"]).mean().round(2))

              precision    recall  f1-score   support

          정상      0.150     0.120     0.133        50
          사기      0.267     0.320     0.291        50

    accuracy                          0.220       100
   macro avg      0.208     0.220     0.212       100
weighted avg      0.208     0.220     0.212       100


=== 유형별 정답률 ===
type
사기_가족사칭    0.53
사기_기타      0.33
사기_로맨스     0.00
사기_부업      0.40
사기_중고거래    0.20
사기_지인사칭    0.30
사기_투자      0.00
정상_공공      0.10
정상_금융알림    0.00
정상_생활      0.07
정상_인증      0.00
정상_지인      0.80
dtype: float64
